# 20. 제품 유형 그룹 분리 기반 일반화 성능 평가

기존 19번 실험의 144개 평가 사례를 사용하되, HS 10단위가 아닌 **HS 6단위**를 예측 대상으로 설정합니다.

이번 실험은 동일한 세부 제품 유형이 학습·평가 데이터에 함께 등장하지 않도록 제품 유형을 의미 기반 상위 제품군으로 정규화하고, **Leave-One-Product-Family-Out** 방식으로 한 제품군 전체를 순서대로 평가 데이터로 제외합니다.

비교 모델:
- TF-IDF + LinearSVC
- SBERT 제품정보만
- SBERT 제품정보 + 과거 판정요지
- SBERT 해설서만
- SBERT 제품정보 + 해설서
- SBERT 제품정보 + 과거 판정요지 + 해설서

주의: 제품군은 Product Type을 바탕으로 사전에 정의한 의미 기반 그룹입니다. 이 그룹화는 HS 코드 정답을 참조해 만들지 않으며, 각 HS 6단위 코드가 둘 이상의 제품군에 존재하는지 검증합니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy

In [ ]:
import re, json, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from kiwipiepy import Kiwi
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
DATA_FILE=quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
KB_FILE='hs_knowledge_base.jsonl'
df=pd.read_csv(BASE+DATA_FILE,encoding='utf-8-sig').fillna('')
kb=pd.read_json(BASE+KB_FILE,lines=True).fillna('')
INPUT_COLUMNS=['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET='HS Code'
SUMMARY='Preprocessed Ruling Summary'
for col in INPUT_COLUMNS+[TARGET,SUMMARY,'Product Type']: df[col]=df[col].astype(str).str.strip()
df=df[df[TARGET].ne('')].copy()
df['duplicate_group']=pd.util.hash_pandas_object(df[INPUT_COLUMNS],index=False).astype(str)
class_counts=df[TARGET].value_counts()
eligible_labels=class_counts[class_counts>=4].index
eval_df=df[df[TARGET].isin(eligible_labels)].copy()
group_counts=eval_df.groupby(TARGET)['duplicate_group'].nunique()
eligible_labels=group_counts[group_counts>=4].index
eval_df=eval_df[eval_df[TARGET].isin(eligible_labels)].reset_index(drop=True)
eval_df['hs_digits']=eval_df[TARGET].map(lambda x:re.sub(r'\D','',str(x)))
eval_df['hs6']=eval_df['hs_digits'].str[:6]
eval_df['hs4']=eval_df['hs_digits'].str[:4]
eval_df['query_text']=eval_df[INPUT_COLUMNS].agg(' '.join,axis=1).str.strip()
eval_df['model_text']='[PRODUCT] '+eval_df['Product Name']+' [INGREDIENTS_KR] '+eval_df['Main Ingredients (KR)']+' [INGREDIENTS_EN] '+eval_df['Main Ingredients (EN)']+' [FORM] '+eval_df['Product Form and Packaging']+' [USE] '+eval_df['Intended Use']
print('평가 사례:',len(eval_df),'| HS 6단위:',eval_df['hs6'].nunique(),'| 원본 Product Type:',eval_df['Product Type'].nunique())

## 1. 의미 기반 제품군 정의

원본 Product Type은 109종으로 세분되어 있고 단일 사례인 유형이 많습니다. 따라서 단순 Product Type 단위 분리 대신, 유사한 제품을 상위 제품군으로 묶습니다.

제품군 정의는 HS 코드가 아니라 제품의 형태·용도·제품 유형을 기준으로 합니다. 아래 규칙은 모든 평가 사례에 제품군을 부여하며, 매핑되지 않은 유형이 있으면 실험을 중단합니다.

In [ ]:
def assign_product_family(value):
    x=str(value).strip()
    if x in {'아이 메이크업 리무버','눈·입술 메이크업 제거'}: return 'eye_makeup_remover'
    if '피내 주사용' in x or '필러' in x: return 'injectable_filler'
    if '향을 첨가한 종이' in x: return 'perfumed_paper'
    if '방향' in x: return 'vehicle_air_freshener' if '차량' in x else 'indoor_air_freshener'
    if '치과' in x or '불소 도포' in x or '산부식' in x: return 'dental_fluoride' if '불소' in x else 'dental_etchant'
    if '목욕' in x: return 'bath_foam' if '거품' in x else 'bath_oil_tablet'
    if '샴푸' in x: return 'shampoo_combo' if '겸용' in x else 'shampoo'
    if '염모' in x or '염색' in x or '컬러스프레이' in x: return 'hair_dye'
    if '헤어' in x or '두발' in x or '두피' in x: return 'hair_care_styling'
    if '립' in x or '입술' in x: return 'lip_balm' if '밤' in x else 'lip_exfoliant' if '스크럽' in x else 'lip_makeup'
    if '속눈썹' in x: return 'eyelash_care'
    if '마스카라' in x or '아이라이너' in x or '아이브로우' in x or '눈 화장' in x: return 'eye_makeup'
    if '볼연지' in x or '블러셔' in x: return 'blush'
    if '파우더' in x or '아이섀도' in x: return 'face_powder'
    if '부직포' in x or '물휴지' in x or '마스크 팩' in x or '마스크팩' in x:
        return 'cosmetic_wipes' if '클렌징' in x or '물휴지' in x else 'impregnated_nonwoven_mask'
    if '외음부' in x or '피어싱' in x or '이어홀' in x: return 'personal_cleansing'
    if '영유아' in x: return 'baby_skincare'
    if '클렌징' in x: return 'skincare_cleansing'
    if '선크림' in x or '자외선' in x: return 'suncare'
    if '팩' in x or '패치' in x or '마스크' in x: return 'skincare_mask_patch'
    if '기능성' in x: return 'functional_skincare'
    return 'basic_skincare'

eval_df['product_family']=eval_df['Product Type'].map(assign_product_family)
if eval_df['product_family'].isna().any(): raise ValueError('제품군 미분류 사례 존재')
family_audit=eval_df.groupby('product_family').agg(n=('hs6','size'),product_types=('Product Type','nunique'),hs6_codes=('hs6',lambda x:', '.join(sorted(x.unique())))).sort_values('n',ascending=False)
display(family_audit)
display(pd.crosstab(eval_df['product_family'],eval_df['hs6']))

## 2. 그룹 분리 가능성 검증

각 HS 6단위 코드가 최소 두 개 이상의 제품군에 존재해야 합니다. 그래야 한 제품군을 통째로 평가에서 제외하더라도 해당 HS 6단위 코드가 학습 데이터에 남아 있어 분류 문제로 평가할 수 있습니다.

이 조건을 만족하지 않는 코드는 이번 일반화 평가 대상에서 임의로 제외하지 않고, 데이터 부족으로 별도 보고합니다.

In [ ]:
label_family_counts=eval_df.groupby('hs6')['product_family'].nunique().sort_values()
display(label_family_counts.rename('제품군 수').to_frame())
unsupported=label_family_counts[label_family_counts<2]
if len(unsupported): raise ValueError(f'둘 이상의 제품군에 존재하지 않는 HS 6단위 코드: {unsupported.to_dict()}')
print('제품군 수:',eval_df['product_family'].nunique(),'| 모든 HS 6단위 코드가 2개 이상 제품군에 존재함')

## 3. 임베딩 및 검색 지식베이스 준비

E5 임베딩은 사전학습 모델을 사용합니다. 임베딩을 미리 계산하더라도 각 평가 Fold에서는 해당 제품군의 문서를 검색 대상에서 제외하므로, 평가 제품의 판정요지가 학습 검색에 유입되지 않습니다.

HS 해설서 문서는 HS 코드가 명시된 문서만 후보 코드 순위 산출에 사용합니다. 문서의 호·소호 코드와 예측 대상 HS 6단위는 숫자 접두어 관계로 연결합니다.

In [ ]:
kiwi=Kiwi()
STOPWORDS={'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
def remove_stopword_morphs(text):
    text=str(text); chars=list(text)
    for token in kiwi.tokenize(text):
        if token.form in STOPWORDS:
            for i in range(token.start,token.start+token.len): chars[i]=''
    return ''.join(chars)

for col in ['document_text','hs_code','rule_type','source_sheet']:
    if col not in kb.columns: kb[col]=''
    kb[col]=kb[col].fillna('').astype(str).str.strip()
kb['hs_digits']=kb['hs_code'].map(lambda x:re.sub(r'\D','',str(x)))
kb_code=kb[kb['document_text'].ne('') & kb['hs_digits'].ne('')].reset_index(drop=True)
kb_code['search_text']=(kb_code['document_text']+' '+kb_code['rule_type']+' '+kb_code['source_sheet']).str.strip()
kb_code['search_text_sw']=kb_code['search_text'].map(remove_stopword_morphs)

MODEL_NAME='intfloat/multilingual-e5-base'
sbert=SentenceTransformer(MODEL_NAME)
def encode_docs(texts):
    return sbert.encode(['passage: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def encode_queries(texts):
    return sbert.encode(['query: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)

query_emb=encode_queries(eval_df['query_text'].tolist())
product_emb=encode_docs(eval_df['query_text'].tolist())
ruling_text=(eval_df['query_text']+' '+eval_df[SUMMARY].map(remove_stopword_morphs)).str.strip()
product_ruling_emb=encode_docs(ruling_text.tolist())
kb_emb=encode_docs(kb_code['search_text_sw'].tolist())
labels=sorted(eval_df['hs6'].unique())
print('제품 사례 임베딩:',product_emb.shape,'| 해설서 임베딩:',kb_emb.shape,'| 장치:',sbert.device)

## 4. HS 6단위 후보 순위 함수

In [ ]:
def rank_case_codes(scores,train_codes):
    tmp=pd.DataFrame({'hs6':np.asarray(train_codes).astype(str),'score':np.asarray(scores)})
    return tmp.groupby('hs6')['score'].max().sort_values(ascending=False)

def code_compatible(doc_code,target_code):
    a=re.sub(r'\D','',str(doc_code)); b=re.sub(r'\D','',str(target_code))
    return bool(a and b and (a.startswith(b) or b.startswith(a)))

def rank_kb_codes(scores):
    values={}
    for j,doc_code in enumerate(kb_code['hs_digits']):
        for label in labels:
            if code_compatible(doc_code,label): values[label]=max(values.get(label,-np.inf),float(scores[j]))
    return pd.Series(values,dtype=float).sort_values(ascending=False)

def rrf_rank(case_rank,kb_rank,k=60):
    case_pos={str(c):i+1 for i,c in enumerate(case_rank.index)}
    kb_pos={str(c):i+1 for i,c in enumerate(kb_rank.index)}
    values={c:1/(k+case_pos.get(c,100000))+1/(k+kb_pos.get(c,100000)) for c in labels}
    return pd.Series(values).sort_values(ascending=False)

def top_codes(rank,n=3): return [str(x) for x in rank.index[:n]]

def make_vectorizer():
    return FeatureUnion([
        ('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),
        ('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))
    ])

## 5. Leave-One-Product-Family-Out 평가

한 제품군 전체를 평가 데이터로 두고 나머지 제품군으로 학습합니다. 따라서 같은 제품군에 속한 제품이 학습과 평가에 동시에 들어가지 않습니다.

모든 HS 6단위 코드가 둘 이상의 제품군에 나타나는지 앞 단계에서 확인했으므로, 각 반복에서 평가 대상 코드가 학습 데이터에 남아 있는지도 추가 검증합니다.

In [ ]:
logo=LeaveOneGroupOut()
splits=list(logo.split(eval_df,eval_df['hs6'],groups=eval_df['product_family']))
prediction_rows=[]
fold_rows=[]
for fold,(tr,te) in enumerate(splits,1):
    train,test=eval_df.iloc[tr].copy(),eval_df.iloc[te].copy()
    train_labels=set(train['hs6'])
    if not set(test['hs6']).issubset(train_labels): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS 6단위 코드')
    fold_family=test['product_family'].iloc[0]
    fold_rows.append({'fold':fold,'heldout_family':fold_family,'train_n':len(tr),'test_n':len(te),'hs6_train_coverage':len(train_labels)})
    vectorizer=make_vectorizer()
    x_train=vectorizer.fit_transform(train['model_text'])
    x_test=vectorizer.transform(test['model_text'])
    svm=LinearSVC(C=1.0,class_weight='balanced').fit(x_train,train['hs6'])
    svm_scores=svm.decision_function(x_test)
    for local_i,global_i in enumerate(te):
        svm_rank=pd.Series(svm_scores[local_i],index=svm.classes_).sort_values(ascending=False)
        prediction_rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':fold_family,'true_hs6':eval_df.iloc[global_i]['hs6'],'true_hs4':eval_df.iloc[global_i]['hs4'],'model':'TFIDF_LinearSVC','predicted_hs6':str(svm_rank.index[0]),'top3_candidates':top_codes(svm_rank)})
    train_idx=np.asarray(tr)
    train_codes=train['hs6'].to_numpy()
    for global_i in te:
        q=query_emb[global_i]
        case_product=rank_case_codes(product_emb[train_idx]@q,train_codes)
        case_ruling=rank_case_codes(product_ruling_emb[train_idx]@q,train_codes)
        kb_rank=rank_kb_codes(kb_emb@q)
        ranks={
            'SBERT_product_only':case_product,
            'SBERT_product_plus_ruling':case_ruling,
            'SBERT_KB_only':kb_rank,
            'SBERT_product_only_plus_KB':rrf_rank(case_product,kb_rank),
            'SBERT_product_plus_ruling_plus_KB':rrf_rank(case_ruling,kb_rank)
        }
        for model_name,rank in ranks.items():
            prediction_rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':fold_family,'true_hs6':eval_df.iloc[global_i]['hs6'],'true_hs4':eval_df.iloc[global_i]['hs4'],'model':model_name,'predicted_hs6':str(rank.index[0]) if len(rank) else '', 'top3_candidates':top_codes(rank)})

predictions=pd.DataFrame(prediction_rows)
fold_audit=pd.DataFrame(fold_rows)
display(fold_audit)
print('제품군 분리 수:',len(splits),'| OOF 예측:',len(predictions),'| 모델별 예측 수:',predictions.groupby('model').size().to_dict())

## 6. HS 4단위·6단위 성능 평가

예측 목표는 HS 6단위입니다. HS 4단위는 필수 일치 조건으로 별도 계산하고, HS 10단위 일치 여부는 평가하지 않습니다.

Top-3는 상위 3개 HS 6단위 후보 중 정답이 포함되는 비율입니다.

In [ ]:
def metric_summary(part):
    yt=part['true_hs6'].astype(str)
    yp=part['predicted_hs6'].astype(str)
    top3=np.mean([str(t) in [str(x) for x in c] for t,c in zip(yt,part['top3_candidates'])])
    return {
        'n':len(part),
        'hs4_accuracy':np.mean(yt.str[:4].to_numpy()==yp.str[:4].to_numpy()),
        'hs6_accuracy':accuracy_score(yt,yp),
        'macro_f1_hs6':f1_score(yt,yp,labels=labels,average='macro',zero_division=0),
        'weighted_f1_hs6':f1_score(yt,yp,average='weighted',zero_division=0),
        'top3_hs6_accuracy':top3
    }

summary=pd.DataFrame([{'model':name,**metric_summary(part)} for name,part in predictions.groupby('model')]).sort_values('hs6_accuracy',ascending=False)
display(summary.round(4))
family_metrics=predictions.groupby(['model','heldout_family']).apply(lambda x:pd.Series(metric_summary(x)),include_groups=False).reset_index()
display(family_metrics.sort_values(['model','heldout_family']))
display(pd.crosstab(predictions['heldout_family'],predictions['true_hs6']))

## 7. 결과 저장

OOF 예측, 전체 모델 비교, 제품군별 성능, Fold 감사표, 제품군 분포를 저장합니다.

In [ ]:
OUT=Path('/content/hs_product_family_generalization')
OUT.mkdir(parents=True,exist_ok=True)
predictions.to_csv(OUT/'20_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'20_model_comparison.csv',index=False,encoding='utf-8-sig')
family_metrics.to_csv(OUT/'20_family_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv(OUT/'20_fold_audit.csv',index=False,encoding='utf-8-sig')
family_audit.to_csv(OUT/'20_product_family_audit.csv',encoding='utf-8-sig')
with zipfile.ZipFile('/content/20_product_family_generalization_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob('*.csv'): z.write(f,arcname=f.name)
print('저장 완료: /content/20_product_family_generalization_results.zip')


## 해석 시 주의사항

- 이번 평가는 무작위 분할보다 어려운 제품군 단위 일반화 평가입니다.
- 제품군 정의가 결과에 영향을 줄 수 있으므로, 제품군 매핑표를 연구 방법에 공개해야 합니다.
- 일부 제품군의 표본 수가 작으므로 제품군별 정확도는 참고용으로 해석하고, 전체 OOF 결과를 중심으로 비교합니다.
- HS 해설서 지식베이스는 고정된 외부 자료이므로 모든 Fold에서 동일하게 사용합니다.
- RRF는 비교 기준선입니다. 해설서 규칙의 의미(포함·제외 등)를 법적 조건에 맞춰 적용하는 후속 실험과는 구분해야 합니다.